<a href="https://colab.research.google.com/github/Computercoder125/AI/blob/main/HW2_dataprep_questions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Sean Gor
Assignment 2 Report

1. What challenges did you encounter when loading and inspecting a raw external dataset from Kaggle compared to pre-packaged datasets?

Unlike pre-packaged datasets that come pre-cleaned with documented feature names, the raw Kaggle CSV (9,537 rows, 10 columns) required manual inspection to understand what I was working with. I used df.columns, df.info(), and df.describe() to confirm data types (a mix of int64, float64, and object), identify session_id as a non-informative identifier, and check that the target column attack_detected was present and binary. I also ran into a practical issue during development: I originally dropped session_id without an errors='ignore' safeguard, and re-running that cell out of order caused a KeyError because the column no longer existed. This taught me to make cleaning steps idempotent (df.drop(columns=['session_id'], errors='ignore')) and to do a full Restart & Run All before treating a notebook as finished — a step that isn't necessary with pre-packaged, static datasets since they don't get modified in place as you explore them.

2. How did you determine which columns required missing value imputation versus other data cleaning methods, and what strategy did you choose?

I quantified missingness with df.isnull().sum() and as percentages, and found the dataset had 0.0% missing values across all 10 columns — this is a real, clean dataset with no gaps to fill. Even so, I implemented a structured imputation strategy so the pipeline would be robust to missing data if the dataset were ever refreshed: median imputation for numerical columns (network_packet_size, session_duration, login_attempts, ip_reputation_score, failed_logins, unusual_time_access) since median is robust to skew and outliers, and mode imputation for categorical columns (protocol_type, encryption_used, browser_type) since mode is the standard default for fill-in on categorical data. Because there were no actual missing values, this step had no visible effect on the data in this run, but it demonstrates the correct approach and would activate automatically going forward.

3. Explain how the Scikit-Learn ColumnTransformer and Pipeline streamline the data preparation process and prevent data leakage across preprocessing steps.

After splitting the data into X (9 features) and y (attack_detected), I separated columns into numerical_features (6 columns) and categorical_features (3 columns: protocol_type, encryption_used, browser_type). The ColumnTransformer let me apply StandardScaler to the numerical columns and OneHotEncoder(handle_unknown='ignore') to the categorical columns in a single fit/transform step, producing a combined output of shape (9537, 16) — the 6 scaled numerical columns plus 10 one-hot encoded categorical columns. Wrapping this in a Pipeline bundles the whole transformation into one reusable object. This matters for preventing data leakage because when .fit_transform() is called, the scaler's mean/std and the encoder's known categories are learned only from the data passed into that call. In a real train/test split, you'd call .fit() only on the training set and .transform() (not .fit()) on the test set — meaning test data is transformed using statistics learned exclusively from training data, so no information about the test distribution leaks into preprocessing.

Based on your correlation matrix and heatmap, which features show the strongest relationships with the target variable (attack_detected), and what operational insights do these relationships reveal?

Among the numerical features, failed_logins had the strongest correlation with attack_detected (0.36) — the highest in the matrix — followed by login_attempts (0.28) and ip_reputation_score (0.21). These three align with well-known intrusion patterns: repeated failed logins and high login attempt counts are classic brute-force/credential-stuffing signatures, and a low IP reputation score suggests traffic from known-bad address ranges. In contrast, network_packet_size (-0.007), session_duration (0.04), and unusual_time_access (0.009) were all essentially uncorrelated with the target, meaning payload size, session length, and off-hours access carry little to no predictive signal on their own in this dataset.

I also extended the analysis to the encoded categorical features, since the required correlation matrix only covers numerical columns. The standout result was browser_type_Unknown, with a correlation of 0.135 — the strongest single relationship found anywhere in the dataset, numerical or categorical — well above every specific protocol or named browser (all of which fell between -0.04 and 0.01).

Based on this, if I were prioritizing features for future monitoring or modeling, I would focus on four in this order:

failed_logins (0.36) — the strongest overall signal, and the most operationally actionable (a threshold-based alert on repeated failures is cheap to implement).
login_attempts (0.28) — closely related to failed_logins and reinforces the brute-force pattern; worth tracking together rather than independently.
browser_type_Unknown (0.135) — the strongest categorical signal and third-strongest overall; flags spoofed or non-standard clients, a distinct attack vector from the login-based signals above.
ip_reputation_score (0.21) — moderately predictive and useful as a complementary signal, especially in combination with the login-based features, since a low-reputation IP combined with high failed logins is a stronger compound indicator than either alone.

I would deprioritize network_packet_size, session_duration, unusual_time_access, and the specific protocol/browser/encryption categories (TCP/UDP, AES/DES, Chrome/Firefox/Safari/Edge) — none showed meaningful correlation with attacks, and including them in a future model would likely add noise rather than useful signal.

Categorical Feature Correlation Analysis

The assignment's required correlation matrix and heatmap only cover numerical features. Since three of the dataset's categorical features (protocol_type, encryption_used, browser_type) are described in the assignment as capturing "communication context and client environment" relevant to "protocol-specific anomalies," I extended the analysis to check whether these categorical values also correlate with attack_detected (shown in the Jupyter Notebook I execute the tasks in), and shown below:

In [ ]:
browser_type_Unknown    0.134630
browser_type_Chrome    -0.040087
protocol_type_ICMP     -0.016619
browser_type_Safari    -0.013289
browser_type_Firefox   -0.010556
encryption_used_AES    -0.008306
encryption_used_DES     0.008306
browser_type_Edge      -0.008057
protocol_type_UDP       0.007903
protocol_type_TCP       0.000643
Name: attack_detected, dtype: float64